# Elite replay EDA

Visualization-only rendering of the tracked tables produced by `scripts/build_elite_eda.py`. Notebook-authored public descriptions are not treated as pinned-runtime measurements. Empty panels are intentional when compatible normalized evidence is unavailable.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
analysis_dir = root / "replays" / "analysis"
daily = pd.read_csv(analysis_dir / "elite_daily.csv")
episodes = pd.read_csv(analysis_dir / "elite_episode_summary.csv")
sources = pd.read_csv(analysis_dir / "elite_source_comparison.csv")
coverage = pd.read_csv(analysis_dir / "elite_coverage_gap.csv")
compatibility = pd.read_csv(analysis_dir / "elite_compatibility.csv")
quarantine = pd.read_csv(analysis_dir / "elite_quarantine.csv")
pd.DataFrame({
    "table": ["daily", "episodes", "sources", "coverage", "compatibility", "quarantine"],
    "rows": [len(daily), len(episodes), len(sources), len(coverage), len(compatibility), len(quarantine)],
})

In [ ]:
def empty_panel(axis, title):
    axis.set_title(title)
    axis.text(0.5, 0.5, "No compatible measured rows", ha="center", va="center", transform=axis.transAxes)
    axis.set_axis_off()

## 1. Daily capital, land, and labor trajectories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
views = (("end_money", "End-of-day bank"), ("mean_land_count", "Mean land count"), ("mean_active_hands", "Mean active hands"))
for axis, (column, title) in zip(axes, views):
    if daily.empty:
        empty_panel(axis, title)
    else:
        sns.lineplot(data=daily, x="day", y=column, hue="source_family", style="source_group", marker="o", ax=axis)
        axis.set_title(title)
fig.suptitle("Daily capital and expansion by source family", y=1.03)
fig.tight_layout()

## 2. Crop and herd composition over time

In [ ]:
def composition_frame(column, asset_type):
    rows = []
    for row in daily.itertuples(index=False):
        values = json.loads(getattr(row, column) or "{}")
        rows.extend({"day": row.day, "source_family": row.source_family, "source_group": row.source_group, "asset_type": asset_type, "asset": asset, "tile_turns": count} for asset, count in values.items())
    return pd.DataFrame(rows)

composition = pd.concat([composition_frame("crop_counts", "crop"), composition_frame("animal_counts", "animal")], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for axis, asset_type in zip(axes, ("crop", "animal")):
    subset = composition[composition.get("asset_type", pd.Series(dtype=str)) == asset_type] if not composition.empty else composition
    if subset.empty:
        empty_panel(axis, f"{asset_type.title()} composition")
    else:
        sns.lineplot(data=subset, x="day", y="tile_turns", hue="asset", style="source_family", marker="o", ax=axis)
        axis.set_title(f"{asset_type.title()} composition")
fig.tight_layout()

## 3. Action allocation and travel/logistics shares

In [ ]:
share_columns = ["productive_action_share", "travel_action_share", "logistics_action_share", "idle_action_share", "other_action_share"]
fig, axis = plt.subplots(figsize=(12, 5))
if sources.empty:
    empty_panel(axis, "Action allocation by source family")
else:
    action_shares = sources.melt(id_vars=["source_group", "source_family"], value_vars=share_columns, var_name="action_family", value_name="share")
    sns.barplot(data=action_shares, x="source_family", y="share", hue="action_family", ax=axis)
    axis.tick_params(axis="x", rotation=30)
    axis.set_title("Action allocation, including travel and logistics")
fig.tight_layout()

## 4. Sell concentration versus observed next-bank changes

In [ ]:
fig, axis = plt.subplots(figsize=(8, 5))
sell_rows = episodes.dropna(subset=["sell_product_hhi", "observed_bank_delta_on_sell_turns"]) if not episodes.empty else episodes
if sell_rows.empty:
    empty_panel(axis, "Sell concentration and observed next-bank change")
else:
    sns.scatterplot(data=sell_rows, x="sell_product_hhi", y="observed_bank_delta_on_sell_turns", hue="source_family", style="source_group", size="sell_quantity", ax=axis)
    axis.set_title("Descriptive association only—not causal proceeds")
fig.tight_layout()

## 5. Storage pressure and final-window conversion

In [ ]:
window_columns = ["final_8_action_cash_change", "final_22_action_cash_change", "final_48_action_cash_change"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if episodes.empty:
    empty_panel(axes[0], "Storage pressure")
    empty_panel(axes[1], "Final-window cash change")
else:
    sns.barplot(data=episodes, x="source_family", y="storage_pressure_turns", hue="source_group", ax=axes[0])
    windowed = episodes.melt(id_vars=["source_group", "source_family"], value_vars=window_columns, var_name="window", value_name="cash_change").dropna(subset=["cash_change"])
    if windowed.empty:
        empty_panel(axes[1], "Final-window cash change")
    else:
        sns.barplot(data=windowed, x="window", y="cash_change", hue="source_family", ax=axes[1])
    axes[0].tick_params(axis="x", rotation=30)
fig.tight_layout()

## 6. Elite-versus-teacher coverage-gap heatmap

In [ ]:
fig, axis = plt.subplots(figsize=(8, 9))
teacher_gap = coverage[(coverage["source_group"] == "teacher") & coverage["distance"].notna()]
if teacher_gap.empty:
    empty_panel(axis, "Elite-versus-teacher coverage distance")
else:
    heatmap = teacher_gap.groupby(["diagnostic", "distance_type"], as_index=False)["distance"].first().pivot(index="diagnostic", columns="distance_type", values="distance")
    sns.heatmap(heatmap, annot=True, fmt=".3f", cmap="vlag", center=0, ax=axis)
    axis.set_title("Elite baseline versus compatible teacher evidence")
fig.tight_layout()

The decision report is `docs/7_elite_replay_eda.md`. Strategy changes and BC collection remain gated by its compatible-evidence decisions.